# 05 - Gold-Label Study-Level CV Splits

This corrected notebook creates leakage-safe study-level five-fold cross-validation using **all complete gold-labelled studies**, not `train.head(500)`.

Corrected flow:

```text
4407 training studies
        ↓
58 complete gold studies
        ↓
5 study-level folds
        ↓
approximately 11-12 gold studies per fold
```

Every 2.5D instance from a given `StudyInstanceUID` stays in the same fold.

**Inputs needed:**
- Original RSNA dataset
- Output of notebook 04 containing `2p5d_index.csv`

**Outputs:**
- `/kaggle/working/cv_folds.csv`
- `/kaggle/working/2p5d_index_with_folds.csv`
- `/kaggle/working/fold_summary.csv`

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import KFold

DATA_DIR = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
WORK_DIR = "/kaggle/working"

train = pd.read_csv(DATA_DIR + "/train.csv")
train_series = pd.read_csv(DATA_DIR + "/train_series.csv")

label_cols = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture"
]

SEED = 42
N_FOLDS = 5

random.seed(SEED)
np.random.seed(SEED)

print("Total train studies:", len(train))
print("Train series rows:", len(train_series))
print("Target columns:", label_cols)

In [ ]:
# Use all complete gold-labelled studies.
# Do NOT use train.head(500) for final baseline CV.

gold_df = train.dropna(subset=label_cols).copy().reset_index(drop=True)

print("Complete gold-labelled studies:", len(gold_df))
display(gold_df[["StudyInstanceUID"] + label_cols].head())

if len(gold_df) < N_FOLDS:
    raise ValueError("Not enough complete gold-labelled studies for 5-fold CV.")

In [ ]:
# Label distribution across all complete gold-labelled studies.

label_counts = gold_df[label_cols].sum().sort_values(ascending=False)
label_prevalence = (gold_df[label_cols].mean() * 100).sort_values(ascending=False)

summary_labels = pd.DataFrame({
    "positive_count": label_counts,
    "prevalence_percent": label_prevalence
})

display(summary_labels)

plt.figure(figsize=(12, 5))
label_counts.plot(kind="bar")
plt.xlabel("Finding")
plt.ylabel("Positive gold studies")
plt.title("Gold Label Distribution Across Complete Gold Studies")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Create study-level 5-fold CV.
# KFold is used because the gold set is very small and multi-label stratification may be unstable.

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

gold_df["fold"] = -1

for fold, (_, val_idx) in enumerate(kf.split(gold_df)):
    gold_df.loc[val_idx, "fold"] = fold

print("Fold counts:")
print(gold_df["fold"].value_counts().sort_index())

assert (gold_df["fold"] >= 0).all()

In [ ]:
# Fold-level label summary.

fold_summary = []

for fold in range(N_FOLDS):
    fold_df = gold_df[gold_df["fold"] == fold]

    row = {
        "fold": fold,
        "num_studies": len(fold_df)
    }

    for col in label_cols:
        row[col + "_positive"] = int(fold_df[col].sum())

    fold_summary.append(row)

fold_summary = pd.DataFrame(fold_summary)
display(fold_summary)

fold_summary_path = WORK_DIR + "/fold_summary.csv"
fold_summary.to_csv(fold_summary_path, index=False)
print("Saved:", fold_summary_path)

In [ ]:
plt.figure(figsize=(7, 4))
gold_df["fold"].value_counts().sort_index().plot(kind="bar")
plt.xlabel("Fold")
plt.ylabel("Number of gold studies")
plt.title("Study-Level Fold Distribution")
plt.tight_layout()
plt.show()

In [ ]:
# Save gold study fold manifest.

cv_folds = gold_df[["StudyInstanceUID", "fold"] + label_cols].copy()

cv_path = WORK_DIR + "/cv_folds.csv"
cv_folds.to_csv(cv_path, index=False)

print("Saved:", cv_path)
print("Rows:", len(cv_folds))
display(cv_folds.head())

In [ ]:
# Auto-detect 2p5d_index.csv from notebook 04 output added as Kaggle input.
# This avoids hardcoding paths like /kaggle/input/notebooks/.../04-2-5d.

def find_file(filename):
    matches = []
    for root, dirs, files in os.walk("/kaggle/input"):
        if filename in files:
            matches.append(os.path.join(root, filename))
    return matches

p2d_matches = find_file("2p5d_index.csv")

print("2p5d_index.csv candidates:")
for path in p2d_matches:
    print(" -", path)

if not p2d_matches:
    raise FileNotFoundError(
        "2p5d_index.csv not found. Add notebook 04 output as input before running 05."
    )

# Prefer notebook 04 output if multiple matches exist.
preferred = [p for p in p2d_matches if "04" in p or "2-5d" in p.lower() or "2p5d" in p.lower()]
index_path = preferred[0] if preferred else p2d_matches[0]

print("Using index_path:", index_path)

index_2p5d = pd.read_csv(index_path)
print("2.5D index shape:", index_2p5d.shape)
print("2.5D studies:", index_2p5d["StudyInstanceUID"].nunique())
display(index_2p5d.head())

In [ ]:
# Merge fold assignments into every 2.5D sample.
# Inner join ensures only gold-labelled studies used in CV are retained.

index_2p5d_with_folds = index_2p5d.merge(
    cv_folds[["StudyInstanceUID", "fold"]],
    on="StudyInstanceUID",
    how="inner"
)

out_path = WORK_DIR + "/2p5d_index_with_folds.csv"
index_2p5d_with_folds.to_csv(out_path, index=False)

print("Saved:", out_path)
print("2.5D samples with folds:", len(index_2p5d_with_folds))
print("Gold studies with 2.5D samples:", index_2p5d_with_folds["StudyInstanceUID"].nunique())
print("Fold counts in 2.5D index:")
print(index_2p5d_with_folds["fold"].value_counts().sort_index())

display(index_2p5d_with_folds.head())

In [ ]:
# Leakage checks.

# 1. Each gold study appears in exactly one fold.
folds_per_study = cv_folds.groupby("StudyInstanceUID")["fold"].nunique()
print("Maximum folds per gold study:", folds_per_study.max())
assert folds_per_study.max() == 1

# 2. Every 2.5D instance from the same study appears in exactly one fold.
folds_per_study_2p5d = index_2p5d_with_folds.groupby("StudyInstanceUID")["fold"].nunique()
print("Maximum folds per study in 2.5D index:", folds_per_study_2p5d.max())
assert folds_per_study_2p5d.max() == 1

# 3. No 2.5D sample should have missing fold.
missing_folds = index_2p5d_with_folds["fold"].isna().sum()
print("Missing fold values in 2.5D index:", missing_folds)
assert missing_folds == 0

print("Leakage check passed.")

In [ ]:
# Final artifact summary.

print("Final artifacts created:")
print(" -", cv_path)
print(" -", out_path)
print(" -", fold_summary_path)

print("Expected use:")
print("cv_folds.csv: one row per gold study with fold assignment")
print("2p5d_index_with_folds.csv: one row per 2.5D sample with study-level fold")

## Report Note

This notebook implements leakage-safe study-level cross-validation on all complete gold-labelled studies. The fold assignment is attached to the 2.5D index so that all samples from the same `StudyInstanceUID` remain in the same fold. These outputs should be used for gold-only EfficientNet and ViT baselines before adding weak supervision.